# Módulo 03 · Regresión y predicción — Soluciones de los ejercicios

Este notebook es **autocontenido**: reproduce la preparación mínima del notebook del módulo (datos, partición train/test y modelo de referencia) y resuelve los cuatro ejercicios finales con código e interpretación.

> Recuerda: los coeficientes de una regresión ajustada sobre datos observacionales describen **asociaciones**, no efectos causales. Lo repetiremos cuando toque interpretar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, LassoCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")

## Preparación (igual que en el notebook del módulo)

In [ ]:
data = load_diabetes(as_frame=True, scaled=False)
df = data.frame.rename(columns={
    "age": "edad", "sex": "sexo", "bmi": "imc", "bp": "presion",
    "s1": "colesterol", "s2": "ldl", "s3": "hdl", "s4": "col_hdl", "s5": "trigliceridos", "s6": "glucosa",
    "target": "progresion",
})
df["sexo"] = (df["sexo"] == df["sexo"].max()).astype(int)

X, y = df.drop(columns="progresion"), df["progresion"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Entrenamiento: {len(X_train)} pacientes · Test: {len(X_test)} pacientes")

# Modelo completo de referencia (statsmodels, como en la sección 3 del notebook)
ols = sm.OLS(y_train, sm.add_constant(X_train)).fit()
# Modelo Lasso de la sección 5 del notebook
lasso = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0)).fit(X_train, y_train)

## Ejercicio 1 · Modelo reducido: `imc`, `presion` y `trigliceridos`

Ajustamos el modelo reducido y el completo con los **mismos** datos de entrenamiento y los evaluamos en el **mismo** test. Añadimos también el R² por validación cruzada en train, que es más estable que una única partición.

In [ ]:
vars_red = ["imc", "presion", "trigliceridos"]
kf = KFold(5, shuffle=True, random_state=0)

def resumen(cols, nombre):
    m = LinearRegression().fit(X_train[cols], y_train)
    pred = m.predict(X_test[cols])
    cv = cross_val_score(LinearRegression(), X_train[cols], y_train, cv=kf, scoring="r2").mean()
    return {"modelo": nombre, "nº variables": len(cols), "R² CV (train)": cv,
            "R² test": r2_score(y_test, pred), "MAE test": mean_absolute_error(y_test, pred)}

comp = pd.DataFrame([resumen(list(X.columns), "Completo (10 variables)"),
                     resumen(vars_red, "Reducido (imc, presion, trigliceridos)")]).set_index("modelo")
dif_test = comp["R² test"].iloc[1] - comp["R² test"].iloc[0]
dif_cv = comp["R² CV (train)"].iloc[1] - comp["R² CV (train)"].iloc[0]
print(comp.round(3))
print(f"\nCambio de R² al pasar al modelo reducido · test: {dif_test:+.3f} · CV: {dif_cv:+.3f}")

In [ ]:
reducido = sm.OLS(y_train, sm.add_constant(X_train[vars_red])).fit()
print(reducido.summary().tables[1])

**Interpretación.** Sorpresa: en este test el modelo reducido **no pierde** R², incluso gana algo (0,502 frente a 0,485), aunque su MAE es ligeramente peor (42,2 frente a 41,5). La validación cruzada, más estable que una sola partición de 111 pacientes, sí muestra una pequeña pérdida (0,451 frente a 0,480). Conclusión honesta: las diferencias son de unas pocas centésimas y están dentro del ruido de muestreo; **ambos modelos predicen prácticamente igual**. Con 3 variables en lugar de 10:

- Todos sus coeficientes son claramente significativos y tienen el signo esperado (más IMC, más presión y más triglicéridos se asocian con más progresión).
- Evita la multicolinealidad entre analíticas (colesterol, LDL, HDL...) que hace inestables los coeficientes del modelo completo; por ejemplo, en el modelo completo el colesterol total tiene coeficiente **negativo**, algo difícil de explicar a un clínico.

**A un médico le explicaría antes el modelo reducido**: casi la misma capacidad predictiva, coeficientes estables e interpretables y menos variables que medir. El modelo completo solo compensaría si la mejora de precisión tuviera valor clínico y se pudiera validar en otra muestra. En cualquier caso, los coeficientes indican **asociación**, no que reducir el IMC en un punto vaya a reducir la progresión en esa cantidad: para afirmar eso haría falta un diseño causal (ensayo o, como mínimo, un análisis que controle los factores de confusión).

Nota: `trigliceridos` (s5 en el conjunto original) está en **escala logarítmica**; su coeficiente se lee mejor como «por cada aumento de 0,1 en el log (≈ +10,5 % en triglicéridos), la progresión esperada sube coeficiente × 0,1 puntos».

## Ejercicio 2 · Interacción `imc × edad`

Añadimos el producto de ambas variables. Ajustamos sobre el conjunto de entrenamiento, como en la sección 3 del notebook.

In [ ]:
df["imc_edad"] = df.imc * df.edad
Xi_train = df.loc[X_train.index].drop(columns="progresion")
inter = sm.OLS(y_train, sm.add_constant(Xi_train)).fit()
print(inter.summary().tables[1])
b_int, p_int = inter.params["imc_edad"], inter.pvalues["imc_edad"]
print(f"\nCoeficiente de la interacción: {b_int:.3f}  (p = {p_int:.3f})")
print(f"R² ajustado sin interacción: {ols.rsquared_adj:.3f} · con interacción: {inter.rsquared_adj:.3f}")

Con el producto sin centrar, los coeficientes de `imc` y `edad` pasan a significar «efecto cuando la otra variable vale 0» (edad 0 o IMC 0), algo sin sentido. Por eso es buena práctica **centrar** las variables antes de multiplicarlas: así los efectos principales se leen en el paciente medio. La significación de la interacción no cambia.

In [ ]:
Xc_train = X_train.copy()
media_imc, media_edad = X_train.imc.mean(), X_train.edad.mean()
Xc_train["imc_c"] = Xc_train.pop("imc") - media_imc
Xc_train["edad_c"] = Xc_train.pop("edad") - media_edad
Xc_train["imc_c_x_edad_c"] = Xc_train.imc_c * Xc_train.edad_c
inter_c = sm.OLS(y_train, sm.add_constant(Xc_train)).fit()
print(inter_c.summary().tables[1])

# Pendiente del IMC para distintas edades
for edad in [30, 50, 70]:
    pend = inter_c.params["imc_c"] + inter_c.params["imc_c_x_edad_c"] * (edad - media_edad)
    print(f"Edad {edad}: +1 punto de IMC → {pend:+.2f} puntos de progresión esperada")

**Interpretación.** El coeficiente de la interacción mide cuánto cambia la pendiente del IMC por cada año adicional de edad (y, simétricamente, cuánto cambia la pendiente de la edad por cada punto de IMC). Aquí vale 0,051 con **p = 0,36**: no es significativo al 5 %, su intervalo de confianza (−0,06 a 0,16) incluye el 0 y el R² ajustado no cambia (0,504 en ambos casos). Leído literalmente, por cada año de edad la pendiente del IMC aumentaría 0,05 puntos: de unos +5 puntos de progresión por punto de IMC a los 30 años a unos +7 a los 70. Pero esa diferencia es compatible con el ruido: **no hay evidencia** de que la asociación entre IMC y progresión dependa de la edad. Fíjate también en que, sin centrar, el coeficiente de `imc` cae a 3,4 con p = 0,20 (es la pendiente para edad 0); al centrar vuelve a 5,9 con p < 0,001.

Dos cautelas: (1) «no significativo» no equivale a «no existe»: con 331 pacientes la potencia para detectar interacciones es baja; (2) aunque fuera significativa, seguiría siendo una asociación, no un efecto causal del IMC.

## Ejercicio 3 · Error de test de Ridge en función de `alpha`

Para que se vean bien las dos zonas usamos el modelo **polinómico de grado 2 + Ridge** de la sección 5 (65 coeficientes): con `alpha` pequeño se comporta como la polinómica sin regularizar (sobreajuste) y con `alpha` muy grande todos los coeficientes se encogen hacia 0 (infraajuste). Calculamos el error en entrenamiento, en validación cruzada y en test.

*Nota metodológica:* el gráfico con el error de test es didáctico. Para **elegir** `alpha` en un proyecto real se usa la validación cruzada sobre train (curva naranja); el test se reserva para la evaluación final.

In [ ]:
alphas = np.logspace(-2, 4, 40)
filas = []
for a in alphas:
    modelo = make_pipeline(StandardScaler(), PolynomialFeatures(2), Ridge(alpha=a))
    modelo.fit(X_train, y_train)
    cv_mae = -cross_val_score(modelo, X_train, y_train, cv=kf, scoring="neg_mean_absolute_error").mean()
    filas.append({"alpha": a,
                  "MAE train": mean_absolute_error(y_train, modelo.predict(X_train)),
                  "MAE CV": cv_mae,
                  "MAE test": mean_absolute_error(y_test, modelo.predict(X_test))})
curva = pd.DataFrame(filas)
mejor_cv = curva.loc[curva["MAE CV"].idxmin()]
mejor_test = curva.loc[curva["MAE test"].idxmin()]

fig, ax = plt.subplots(figsize=(10, 4.5))
for col in ["MAE train", "MAE CV", "MAE test"]:
    ax.plot(curva.alpha, curva[col], marker="o", ms=3, label=col)
ax.set_xscale("log")
ax.axvline(mejor_cv.alpha, ls="--", c="grey")
ax.axvspan(alphas[0], 1, color="tab:red", alpha=0.08)
ax.axvspan(2000, alphas[-1], color="tab:blue", alpha=0.08)
ax.text(0.012, ax.get_ylim()[1] * 0.97, "zona de SOBREajuste\n(alpha pequeño)", va="top", color="tab:red")
ax.text(2100, ax.get_ylim()[1] * 0.97, "zona de\nINFRAajuste", va="top", color="tab:blue")
ax.set(xlabel="alpha (escala log)", ylabel="MAE", title="Polinómica grado 2 + Ridge: error frente a alpha")
ax.legend(loc="upper center"); plt.tight_layout(); plt.show()

print(curva.iloc[[0, 10, 20, 25, 30, 39]].round(2).to_string(index=False))
print(f"\nalpha óptimo por CV: {mejor_cv.alpha:.1f} (MAE CV {mejor_cv['MAE CV']:.1f}, MAE test {mejor_cv['MAE test']:.1f})")
print(f"alpha con menor MAE de test: {mejor_test.alpha:.1f} (MAE test {mejor_test['MAE test']:.1f})")

**Interpretación.**

- **Zona de sobreajuste (alpha pequeño, a la izquierda):** el error de entrenamiento es el más bajo de toda la curva (MAE ≈ 39), pero el de validación cruzada es claramente peor (≈ 52) y el de test también sube (≈ 44). El modelo memoriza ruido con sus 65 coeficientes.
- **Zona óptima (alpha intermedio, ~10–100):** el error de validación cruzada alcanza su mínimo (alpha ≈ 35, MAE CV ≈ 47) y el de test también es el más bajo (≈ 41), similar al de la regresión lineal de 10 variables (41,5).
- **Zona de infraajuste (alpha muy grande, a la derecha):** los tres errores crecen juntos (≈ 63–64 con alpha = 10.000) y se acercan al MAE de predecir siempre la media (≈ 65): el modelo está tan penalizado que casi no aprende nada.

La distancia entre la curva de train y las de CV/test es la medida visual del sobreajuste: se estrecha conforme aumenta `alpha`.

Observa también que en esta partición el MAE de test es sistemáticamente menor que el de validación cruzada: los 111 pacientes de test resultan algo «más fáciles» que la media. Es un recordatorio de que una única partición train/test tiene bastante variabilidad con muestras pequeñas.

## Ejercicio 4 · Priorizar al 20 % de pacientes

El hospital solo puede hacer seguimiento intensivo al 20 % de los pacientes. Ordenamos los pacientes de test por la progresión **predicha** por el modelo Lasso, seleccionamos el 20 % superior y comprobamos cuántos de ellos están realmente en el 20 % con mayor progresión **observada**.

In [ ]:
n_sel = int(round(0.2 * len(y_test)))
pred_test = pd.Series(lasso.predict(X_test), index=y_test.index)

seleccion_modelo = set(pred_test.nlargest(n_sel).index)
top_real = set(y_test.nlargest(n_sel).index)
captura = len(seleccion_modelo & top_real) / n_sel

# Referencias: selección al azar (esperado = 20 %) y selección solo por IMC
seleccion_imc = set(X_test.imc.nlargest(n_sel).index)
captura_imc = len(seleccion_imc & top_real) / n_sel
rng = np.random.default_rng(0)
capturas_azar = [len(set(rng.choice(y_test.index, n_sel, replace=False)) & top_real) / n_sel for _ in range(2000)]

print(f"Pacientes en test: {len(y_test)} → seleccionamos {n_sel}")
print(f"Captura del verdadero top-20 % con el modelo Lasso: {captura:.0%}")
print(f"Captura ordenando solo por IMC:                       {captura_imc:.0%}")
print(f"Captura esperada al azar:                             {np.mean(capturas_azar):.0%}")

In [ ]:
# ¿Cómo son los pacientes seleccionados frente al resto?
sel = y_test.index.isin(list(seleccion_modelo))
print(f"Progresión media observada · seleccionados: {y_test[sel].mean():.0f} · resto: {y_test[~sel].mean():.0f}")

# Curva de captura para distintos porcentajes de seguimiento
pcts = np.arange(0.05, 1.0001, 0.05)
cap = []
for p in pcts:
    k = int(round(p * len(y_test)))
    cap.append(len(set(pred_test.nlargest(k).index) & set(y_test.nlargest(k).index)) / k)
plt.plot(pcts * 100, np.array(cap) * 100, marker="o", label="modelo Lasso")
plt.plot(pcts * 100, pcts * 100, ls="--", c="grey", label="azar")
plt.axvline(20, c="tab:red", alpha=0.4)
plt.xlabel("% de pacientes con seguimiento"); plt.ylabel("% del top real capturado")
plt.title("Captura de los pacientes de mayor progresión"); plt.legend(); plt.show()

**Interpretación.** De los 22 pacientes con mayor progresión real en test, el modelo Lasso captura **15 (68 %)**, frente al 20 % esperable eligiendo al azar y al 55 % de ordenar solo por IMC: el modelo multiplica por más de tres la eficacia del cribado aleatorio. Los seleccionados tienen una progresión media observada de 238 puntos frente a 123 en el resto. Aun así, una parte importante de los pacientes de mayor progresión **se escaparía**, de modo que el modelo debería servir para **priorizar**, no para excluir a nadie de la atención.

Cautelas para el memorando: el test tiene solo 111 pacientes (el porcentaje tiene un margen de error amplio; con otra semilla de partición podría variar varios puntos), la muestra procede de una única población y el modelo es predictivo, no causal: no dice qué intervención reduciría la progresión.

## Resumen para el memorando (caso práctico)

- **Modelo:** Lasso (o el lineal reducido de 3 variables si se prioriza la interpretabilidad). MAE en test ≈ 41 puntos frente a ≈ 65 prediciendo la media.
- **Variables más influyentes (coeficientes estandarizados):** IMC, triglicéridos (log) y presión arterial, con signo positivo. Son asociaciones, no efectos causales.
- **Priorización:** con seguimiento al 20 %, el modelo captura el 68 % del verdadero top-20 % (azar: 20 %; solo IMC: 55 %).
- **Limitaciones:** 442 pacientes de una sola población, multicolinealidad entre analíticas, ausencia de diseño causal y validación en un único conjunto de test.